# Figures for §7.3 — transfer to CARLA

| output | placeholder it fills |
|---|---|
| `carla-campaign.pdf` / `.png` | the coverage-and-ride figure for §7.3 |
| `carla-campaign-served.csv` / `.md` | the gap table the section quotes |
| `carla-campaign-ride.csv` / `.md` | ride time by town |

**Reads `results/runs` only — it does not need a CARLA server and does not
re-simulate anything.** It is safe to run while a campaign is in flight;
worst case it reports the campaign as incomplete and falls back.

## This notebook has to survive a half-finished campaign

The campaign was re-run after the 2026-08-24 steering change, and steering is
part of a run's identity, so the store holds two campaigns side by side:
`direct-drive-v1` (complete, 150 runs) and `direct-drive-v2` (in progress).

**Pooling them would be wrong** — the change alters how long a leg takes — so
this notebook picks exactly one:

* if `direct-drive-v2` is complete, every figure is built from it;
* otherwise the figures come from `direct-drive-v1`, and **every output is
  stamped PROVISIONAL**, including the figure itself, so a stale chart cannot
  quietly end up in the document.

Re-run it when the campaign finishes; the stamp disappears on its own.

## Why wait time is not plotted

The window is 750 s and passenger patience is 750 s. Anyone who waited long
enough to matter has already cancelled and left the sample, so mean wait is
computed over survivors and sits in a narrow band for every policy in every
town that discriminates at all. **Coverage and ride time are the honest
KPIs here** — §6.3 explains why at length, and plotting wait would invite
exactly the misreading that section warns against.

## Setup

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from cycler import cycler
from matplotlib.lines import Line2D
from scipy import stats

from src.experiment.store import DEFAULT_RUNS_DIR, load_frame

from IPython.display import Markdown, display

# Where the submission's figures are written. Outside the repo, beside the
# document that embeds them, so a figure and the prose quoting it move
# together. Override with AUTOBUS_FIGURES to render somewhere disposable.
FIGURES = Path(os.environ.get(
    "AUTOBUS_FIGURES",
    Path.home() / "Desktop" / "האוניברסיטה הפתוחה" / "סדנה"
    / "submission prep docs" / "figures"))
FIGURES.mkdir(parents=True, exist_ok=True)

# The campaign's palette, so every figure in the submission reads as one set.
SERIES = ["#2a78d6", "#eb6834", "#1baf7a", "#8b5cf6", "#d4a017"]
SURFACE, INK, INK_SOFT, GRID = "#fcfcfb", "#0b0b0b", "#52514e", "#e6e5e1"
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE,
    "savefig.facecolor": SURFACE, "axes.prop_cycle": cycler(color=SERIES),
    "axes.edgecolor": INK_SOFT, "axes.labelcolor": INK_SOFT,
    "axes.titlecolor": INK, "axes.titlesize": 11, "axes.titleweight": "bold",
    "axes.labelsize": 9.5, "axes.spines.top": False,
    "axes.spines.right": False, "grid.color": GRID, "grid.linewidth": 0.8,
    "xtick.color": INK_SOFT, "ytick.color": INK_SOFT, "xtick.labelsize": 9,
    "ytick.labelsize": 9, "legend.frameon": False, "legend.fontsize": 9,
    "figure.dpi": 110,
    # Text stays text in the PDF, so the document can be searched and the
    # figure scales without going fuzzy.
    "pdf.fonttype": 42, "ps.fonttype": 42,
})


def save(figure, name: str) -> None:
    """
    Write one figure as both a vector PDF and a raster PNG.

    The PDF is what belongs in the submission document -- it stays sharp at
    any zoom and its labels remain selectable text. The PNG is for anything
    that cannot embed a PDF, such as slides or a video frame.

    :param figure: The matplotlib figure to write.
    :param name: Base filename, without an extension.
    :return: None.
    """
    figure.tight_layout()
    for suffix in ("pdf", "png"):
        path = FIGURES / f"{name}.{suffix}"
        figure.savefig(path, format=suffix, dpi=300, bbox_inches="tight")
    print(f"wrote {FIGURES / name}.pdf and .png")


def _column_format(values: "pd.Series"):
    """
    Choose one rendering for a whole column.

    Per column rather than per value: a column that renders 46.70 beside a
    bare 50 reads as two different measurements, and a p-value of 2.4e-21
    beside a sample size of 60 needs a different rule again.

    :param values: The column.
    :return: A function from value to text.
    """
    numbers = [v for v in values
               if isinstance(v, (int, float)) and not isinstance(v, bool)
               and v == v]
    if not numbers:
        return lambda v: "—" if v != v else str(v)
    finite = [abs(v) for v in numbers if v]
    if finite and min(finite) < 0.001:
        return lambda v: "—" if v != v else f"{v:.1e}"
    if all(float(v).is_integer() and abs(v) < 1e6 for v in numbers):
        return lambda v: "—" if v != v else f"{int(v):,}"
    return lambda v: "—" if v != v else f"{v:,.2f}"


def as_markdown(frame: pd.DataFrame) -> str:
    """
    Render a table as a Markdown block ready to paste into the document.

    Written out by hand rather than with ``DataFrame.to_markdown``, which
    needs the ``tabulate`` package -- not a dependency worth adding to the
    environment to format six rows.

    :param frame: The table to render. Name its index and columns for the
        reader first: ``identity.world.town`` is a store path, not a
        heading.
    :return: A Markdown table, index included as the first column.
    """
    header = [str(frame.index.name or "")] + [str(c) for c in frame.columns]
    lines = ["| " + " | ".join(header) + " |",
             "|" + "|".join("---" for _ in header) + "|"]
    formats = {c: _column_format(frame[c]) for c in frame.columns}
    for key, row in frame.iterrows():
        cells = [formats[c](row[c]) if isinstance(row[c], (int, float))
                 and not isinstance(row[c], bool) else str(row[c])
                 for c in frame.columns]
        lines.append("| " + " | ".join([str(key)] + cells) + " |")
    return "\n".join(lines) + "\n"


def save_table(frame: pd.DataFrame, name: str) -> None:
    """
    Write one table as CSV and as a Markdown block ready to paste.

    :param frame: The table to write.
    :param name: Base filename, without an extension.
    :return: None.
    """
    frame.to_csv(FIGURES / f"{name}.csv")
    (FIGURES / f"{name}.md").write_text(as_markdown(frame), encoding="utf-8")
    print(f"wrote {FIGURES / name}.csv and .md")


def ci_half_width(values: pd.Series, confidence: float = 0.95) -> float:
    """
    Half-width of the confidence interval on a mean.

    The design review's SS3.2 bar is stated on this quantity, so it is
    computed the same way here as in ``summarize_comparison``.

    :param values: The per-seed observations.
    :param confidence: Confidence level.
    :return: Half-width in the units of ``values``, or nan if n < 2.
    """
    n = values.count()
    if n < 2:
        return float("nan")
    return float(stats.t.ppf(0.5 + confidence / 2, n - 1)
                 * values.std(ddof=1) / np.sqrt(n))


print("figures will be written to:", FIGURES)

## Which campaign is complete?

The table below is also the answer to "is the re-run done yet?" — 10 seeds in
every cell is a finished campaign.

In [ ]:
# The campaign's shape. Anything outside this is a different experiment
# that happens to share the store.
TOWNS = ["Town01", "Town02", "Town03", "Town05", "TAU"]
POLICY_LABELS = ["Fixed route", "Window W750", "Reordering W750"]
CAPACITY, MAX_REQUESTS, INTERVAL_S, SEEDS = 20, 30, 90.0, 10

# Steering is versioned because it changes how long a leg takes, so runs
# made under different steering must never pool. v2 is the 2026-08-24
# change; v1 is the campaign it replaces.
STEERING_CURRENT, STEERING_PREVIOUS = "direct-drive-v2", "direct-drive-v1"


def campaign(steering: str) -> pd.DataFrame:
    """
    Every campaign run recorded under one steering version.

    :param steering: Steering identity tag.
    :return: One row per run.
    """
    frame = load_frame(DEFAULT_RUNS_DIR)
    return frame[
        (frame["identity.world.backend"] == "CarlaBackend")
        & (frame["identity.world.steering"] == steering)
        & (frame["identity.world.town"].isin(TOWNS))
        & (frame["identity.trial.capacity"] == CAPACITY)
        & (frame["identity.trial.max_requests"] == MAX_REQUESTS)
        & (frame["identity.trial.demand_interval_s"] == INTERVAL_S)
        & (frame["identity.policy"].isin(POLICY_LABELS))
    ]


def completeness(runs: pd.DataFrame) -> pd.DataFrame:
    """
    Seeds recorded per town and policy, against the full campaign.

    :param runs: Runs from ``campaign``.
    :return: Towns as rows, policies as columns.
    """
    if runs.empty:
        return pd.DataFrame()
    return (runs.groupby(["identity.world.town", "identity.policy"])
            .size().unstack(fill_value=0)
            .reindex(index=TOWNS, columns=POLICY_LABELS, fill_value=0))


EXPECTED = len(TOWNS) * len(POLICY_LABELS) * SEEDS
current, previous = campaign(STEERING_CURRENT), campaign(STEERING_PREVIOUS)

display(Markdown(f"**Runs recorded, out of {EXPECTED}.**"))
for tag, runs in ((STEERING_CURRENT, current), (STEERING_PREVIOUS, previous)):
    print(f"{tag}: {len(runs)} / {EXPECTED}")
    if not runs.empty:
        display(completeness(runs))

# Which campaign the figures are built from. The current steering wins as
# soon as it is complete; until then the figures come from the campaign that
# actually finished, and every output says so rather than mixing the two.
if len(current) == EXPECTED:
    STEERING, runs, provisional = STEERING_CURRENT, current, False
else:
    STEERING, runs, provisional = STEERING_PREVIOUS, previous, True

STAMP = f"CARLA campaign, steering {STEERING}, {SEEDS} seeds a cell"
if provisional:
    missing = completeness(current)
    absent = [t for t in TOWNS if missing.empty or missing.loc[t].sum() == 0]
    display(Markdown(
        f"> ⚠ **Provisional: built from `{STEERING_PREVIOUS}`.** The current "
        f"steering has {len(current)} of {EXPECTED} runs"
        + (f", and these towns have none yet: **{', '.join(absent)}**"
           if absent else "")
        + ". Re-run this notebook when the campaign finishes and every "
          "output below is replaced. Nothing here pools the two versions."))
else:
    display(Markdown(
        f"> **Current.** All {EXPECTED} runs are recorded under "
        f"`{STEERING_CURRENT}`; these figures are final."))

assert not runs.empty, "no campaign runs found under either steering tag"

## The figure

In [ ]:
TOWN_ORDER_BY = "Fixed route"     # towns sorted by how hard they are
COLOURS = {"Fixed route": SERIES[0], "Window W750": SERIES[1],
           "Reordering W750": SERIES[2]}
GATE = 80.0


def by_town(runs: pd.DataFrame, column: str, scale: float = 1.0):
    """
    One KPI per town and policy, hardest town first.

    :param runs: Campaign runs.
    :param column: KPI column.
    :param scale: Multiplier, 100 for fractions.
    :return: Towns as rows, policies as columns.
    """
    table = runs.pivot_table(index="identity.world.town",
                             columns="identity.policy",
                             values=column) * scale
    table = table.reindex(columns=POLICY_LABELS)
    return table.sort_values(TOWN_ORDER_BY)


served = by_town(runs, "kpi.served_fraction", 100.0)
ride = by_town(runs, "kpi.mean_ride_time")

figure, axes = plt.subplots(1, 2, figsize=(11.8, 4.6),
                            gridspec_kw={"width_ratios": [1.25, 1]})

# --- Left: coverage, the KPI the gate is stated on -----------------------
left = axes[0]
positions = np.arange(len(served))
width = 0.26
for offset, policy in zip((-width, 0.0, width), POLICY_LABELS):
    left.bar(positions + offset, served[policy], width=width,
             color=COLOURS[policy], label=policy)
left.axhline(GATE, ls="--", lw=1.2, color=INK_SOFT)
left.text(len(served) - 0.45, GATE + 1.5, "80% gate", fontsize=8.5,
          color=INK_SOFT, ha="right")
left.set_xticks(positions)
left.set_xticklabels(served.index)
left.set_ylabel("served fraction (%)")
left.set_ylim(0, 108)
left.set_title("Coverage — contenders clear the gate, fixed route does not")
left.grid(axis="y")
left.set_axisbelow(True)
left.legend(fontsize=8.5, ncol=3, loc="upper left")

# --- Right: ride time, the KPI that is not censored ----------------------
# Wait time is deliberately absent: at a 750 s window against 750 s
# patience anyone who waited long has already cancelled, so its mean is a
# survivor statistic and it sits in a narrow band for every arm. Ride time
# is where the routing difference actually shows.
right = axes[1]
for policy in POLICY_LABELS:
    right.plot(ride.index, ride[policy], marker="o", markersize=7, lw=2.0,
               color=COLOURS[policy], label=policy)
right.set_ylabel("mean ride time (s)")
right.set_title("Ride time — the uncensored KPI")
right.grid(axis="y")
right.set_axisbelow(True)
right.tick_params(axis="x", labelrotation=0)

# Name the biggest single saving; it is the number the section quotes.
cut = (ride["Fixed route"] - ride["Reordering W750"])
town = cut.idxmax()
right.annotate(
    f"{town}: {ride.loc[town, 'Fixed route']:.0f} → "
    f"{ride.loc[town, 'Reordering W750']:.0f} s"
    f" ({cut[town] / ride.loc[town, 'Fixed route']:.0%})",
    xy=(list(ride.index).index(town), ride.loc[town, "Reordering W750"]),
    xytext=(0, -34), textcoords="offset points", fontsize=8.5,
    color=INK, ha="center",
    arrowprops=dict(arrowstyle="->", color=INK_SOFT, lw=1.0))

figure.suptitle(STAMP + ("  —  PROVISIONAL" if provisional else ""),
                fontsize=10, color=SERIES[1] if provisional else INK_SOFT,
                y=1.02)
save(figure, "carla-campaign")
plt.show()

## The tables

The gap column is what §7.3 leads with. The L1-vs-L0 test underneath is
reported with its p-value on purpose: the honest claim is that a 10-seed
campaign **cannot resolve** a difference that small, and a table of means
without it reads like a win.

In [ ]:
gap_table = served.copy()
gap_table["gap (R − F)"] = (gap_table["Reordering W750"]
                            - gap_table["Fixed route"])
gap_table = gap_table.sort_values("gap (R − F)", ascending=False)
gap_table.index.name = "Town"
ride.index.name = "Town"
for table in (gap_table, ride):
    table.columns.name = None

display(Markdown(f"**Served fraction (%) by town and policy.** {STAMP}."))
display(gap_table.round(1))
save_table(gap_table.round(1), "carla-campaign-served")

display(Markdown("**Mean ride time (s).**"))
display(ride.round(1))
save_table(ride.round(1), "carla-campaign-ride")

# L1 against L0, paired by (town, seed). Reported with its p-value because
# the honest claim here is that the campaign cannot resolve this difference,
# and a table of means alone would invite the opposite reading.
paired = runs.pivot_table(
    index=["identity.world.town", "identity.seed"],
    columns="identity.policy", values="kpi.served_fraction").dropna() * 100
difference = paired["Reordering W750"] - paired["Window W750"]
_, p_value = stats.ttest_rel(paired["Reordering W750"],
                             paired["Window W750"])
print(f"L1 − L0 pooled: {difference.mean():+.2f} points, "
      f"p = {p_value:.3f}, n = {len(difference)} town-seed pairs")
print("under-powered at this sample size -- do not claim a CARLA-side L1 win"
      if p_value >= 0.05 else "significant at this sample size")

# The claims section 7.3 makes about this table.
contenders = gap_table[["Window W750", "Reordering W750"]]
print(f"\ncontenders clearing the {GATE:.0f}% gate: "
      f"{int((contenders >= GATE).all(axis=1).sum())} of "
      f"{len(gap_table)} towns")
print(f"towns where fixed route fails it: "
      f"{int((gap_table['Fixed route'] < GATE).sum())} of {len(gap_table)}")
inert = gap_table.index[gap_table["Fixed route"] >= 99.0].tolist()
if inert:
    print(f"inert town(s) -- fixed route already serves ~100%, so coverage "
          f"cannot discriminate: {', '.join(inert)}")